# EDA catalogo de ofertas

Flujo:

1. Carga de datos.
2. Revision de nulos originales.
3. Recodificacion de nulos estructurales.
4. EDA del catalogo.

El catalogo es una lista pequena de ofertas, por eso se priorizan conteos y revision de atributos, no estadistica poblacional.

In [1]:
# Graficar distribucion
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)
sns.set_theme(style="whitegrid")

In [2]:
import os
from pathlib import Path

# Configuración de ruta de datos (dinámica y sobreescribible vía variable de entorno DATA_DIR)
DATA_DIR = Path(os.getenv("DATA_DIR", "../Data")).resolve()

# Cargar dataset
catalogo = pd.read_csv(DATA_DIR / "catalogo_ofertas_entrega.csv")

In [3]:
# Ver dimensiones del dataset
catalogo.shape

(22, 11)

In [4]:
# Ver primeras filas del dataset
catalogo.head()

,oferta_id,nombre_oferta,tipo_oferta,segmento_objetivo,es_movistar_total,precio_mensual,ahorro_pct,gb_incluidos,cluster_hogar,descripcion_bundle,descripcion_corta
0,OF001,Plan Movil Basico 10GB,plan_movil,movil,False,39.9,0,10,NaN,NaN,Plan Movil Basico 10GB - 10GB - S/ 39.9
1,OF002,Plan Movil Plus 25GB,plan_movil,movil,False,59.9,0,25,NaN,NaN,Plan Movil Plus 25GB - 25GB - S/ 59.9
2,OF003,Plan Movil Max 50GB,plan_movil,movil,False,79.9,0,50,NaN,NaN,Plan Movil Max 50GB - 50GB - S/ 79.9
3,OF004,Plan Movil Ilimitado,plan_movil,movil,False,99.9,0,9999,NaN,NaN,Plan Movil Ilimitado - 9999GB - S/ 99.9
4,OF005,Internet Hogar 100Mb,plan_hogar,hogar,False,89.9,0,0,mono,Internet,Internet Hogar 100Mb - 0GB - S/ 89.9


In [5]:
# Estadisticos generales del dataset completo
catalogo.describe(include="all")


,oferta_id,nombre_oferta,tipo_oferta,segmento_objetivo,es_movistar_total,precio_mensual,ahorro_pct,gb_incluidos,cluster_hogar,descripcion_bundle,descripcion_corta
count,22,22,22,22,22,22.000000,22.000000,22.000000,6,6,22
unique,22,22,6,3,2,NaN,NaN,NaN,3,5,22
top,OF001,Plan Movil Basico 10GB,plan_hogar,movil,False,NaN,NaN,NaN,mono,Internet,Plan Movil Basico 10GB - 10GB - S/ 39.9
freq,1,1,6,9,19,NaN,NaN,NaN,3,2,1
mean,NaN,NaN,NaN,NaN,NaN,83.018182,4.772727,919.681818,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,60.788922,13.136397,2938.753014,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,12.900000,0.000000,0.000000,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,32.400000,0.000000,0.000000,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,74.900000,0.000000,0.000000,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,117.400000,0.000000,28.750000,NaN,NaN,NaN


## 1. Nulos originales

In [6]:
# Revisar nulos originales
catalogo.isna().sum().to_frame("nulos").assign(
    pct=lambda x: (x["nulos"] / len(catalogo) * 100).round(2)
).query("nulos > 0")

,nulos,pct
cluster_hogar,16,72.73
descripcion_bundle,16,72.73


In [7]:
# Revisar distribucion o validacion simple
pd.crosstab(catalogo["cluster_hogar"].isna(), catalogo["tipo_oferta"])

tipo_oferta,equipo,movistar_total,paquete_adicional,plan_hogar,plan_movil,upgrade
cluster_hogar,,,,,,
False,0,0,0,6,0,0
True,3,3,3,0,4,3


In [8]:
# Revisar distribucion o validacion simple
pd.crosstab(catalogo["descripcion_bundle"].isna(), catalogo["tipo_oferta"])

tipo_oferta,equipo,movistar_total,paquete_adicional,plan_hogar,plan_movil,upgrade
descripcion_bundle,,,,,,
False,0,0,0,6,0,0
True,3,3,3,0,4,3


**Hallazgo:** `cluster_hogar` y `descripcion_bundle` no aplican a todas las ofertas. Por eso esos nulos se tratan como `no_aplica`, no como error.

## 2. Recodificacion de nulos estructurales

In [9]:
# Ejecutar analisis
ids = ["oferta_id"]

for col in catalogo.select_dtypes(include="object").columns:
    if col not in ids:
        catalogo[col] = catalogo[col].str.strip().str.lower()

In [10]:
# Recodificacion de nulos estructurales
reporte_nulos = []

mask = catalogo["cluster_hogar"].isna()
cond = catalogo["tipo_oferta"] != "plan_hogar"
catalogo.loc[mask & cond, "cluster_hogar"] = "no_aplica"
reporte_nulos.append(["cluster_hogar", mask.sum(), "tipo_oferta != plan_hogar", "no_aplica", (mask & cond).sum(), (mask & ~cond).sum()])

mask = catalogo["descripcion_bundle"].isna()
cond = catalogo["tipo_oferta"] != "plan_hogar"
catalogo.loc[mask & cond, "descripcion_bundle"] = "no_aplica"
reporte_nulos.append(["descripcion_bundle", mask.sum(), "tipo_oferta != plan_hogar", "no_aplica", (mask & cond).sum(), (mask & ~cond).sum()])

reporte_nulos = pd.DataFrame(reporte_nulos, columns=[
    "columna", "nulos_originales", "regla_usada", "categoria_creada",
    "cantidad_recodificada", "nulos_problematicos_restantes"
])
reporte_nulos

,columna,nulos_originales,regla_usada,categoria_creada,cantidad_recodificada,nulos_problematicos_restantes
0,cluster_hogar,16,tipo_oferta != plan_hogar,no_aplica,16,0
1,descripcion_bundle,16,tipo_oferta != plan_hogar,no_aplica,16,0


**Hallazgo:** los nulos quedaron recodificados como `no_aplica`, manteniendo su significado de negocio.

## 3. EDA del catalogo

In [11]:
# Revisar distribucion o validacion simple
catalogo["tipo_oferta"].value_counts(dropna=False).to_frame("n").assign(
    pct=lambda x: (x["n"] / len(catalogo) * 100).round(2)
)

,n,pct
tipo_oferta,,
plan_hogar,6,27.27
plan_movil,4,18.18
upgrade,3,13.64
equipo,3,13.64
paquete_adicional,3,13.64
movistar_total,3,13.64


**Hallazgo:** el catalogo tiene pocas ofertas por tipo. Conviene revisar tambien a nivel de `nombre_oferta`.

In [12]:
# Revisar distribucion o validacion simple
catalogo["segmento_objetivo"].value_counts(dropna=False).to_frame("n").assign(
    pct=lambda x: (x["n"] / len(catalogo) * 100).round(2)
)

,n,pct
segmento_objetivo,,
movil,9,40.91
hogar,8,36.36
ambos,5,22.73


**Hallazgo:** el segmento objetivo indica si la oferta apunta a movil, hogar o ambos.

In [13]:
# Revisar distribucion o validacion simple
catalogo["cluster_hogar"].value_counts(dropna=False).to_frame("n").assign(
    pct=lambda x: (x["n"] / len(catalogo) * 100).round(2)
)

,n,pct
cluster_hogar,,
no_aplica,16,72.73
mono,3,13.64
duo,2,9.09
trio,1,4.55


**Hallazgo:** `no_aplica` separa ofertas que no son de hogar de los clusters reales de hogar.

In [14]:
# Revisar distribucion o validacion simple
catalogo["es_movistar_total"].value_counts(dropna=False).to_frame("n").assign(
    pct=lambda x: (x["n"] / len(catalogo) * 100).round(2)
)

,n,pct
es_movistar_total,,
False,19,86.36
True,3,13.64


**Hallazgo:** las ofertas Movistar Total son pocas, pero son prioritarias para el reto.

In [15]:
# Ejecutar analisis
catalogo[["oferta_id", "nombre_oferta", "tipo_oferta", "segmento_objetivo", "precio_mensual", "ahorro_pct", "gb_incluidos", "cluster_hogar"]].sort_values("tipo_oferta")

,oferta_id,nombre_oferta,tipo_oferta,segmento_objetivo,precio_mensual,ahorro_pct,gb_incluidos,cluster_hogar
15,OF016,router wifi 6,equipo,hogar,15.0,0,0,no_aplica
13,OF014,equipo smartphone gama media,equipo,movil,45.0,0,0,no_aplica
14,OF015,equipo smartphone gama alta,equipo,movil,90.0,0,0,no_aplica
21,OF022,movistar total max,movistar_total,ambos,229.9,50,9999,no_aplica
19,OF020,movistar total basico,movistar_total,ambos,149.9,20,30,no_aplica
20,OF021,movistar total plus,movistar_total,ambos,189.9,35,60,no_aplica
18,OF019,paquete roaming internacional,paquete_adicional,movil,29.9,0,5,no_aplica
17,OF018,paquete seguridad digital,paquete_adicional,ambos,12.9,0,0,no_aplica
16,OF017,paquete streaming video,paquete_adicional,ambos,19.9,0,0,no_aplica
4,OF005,internet hogar 100mb,plan_hogar,hogar,89.9,0,0,mono


**Hallazgo:** revisar la tabla de atributos permite comparar precio, ahorro y GB por oferta sin tratar el catalogo como una poblacion numerica grande.

In [16]:
# Ejecutar analisis
catalogo[catalogo["es_movistar_total"] == True][[
    "oferta_id", "nombre_oferta", "precio_mensual", "ahorro_pct", "gb_incluidos"
]]

,oferta_id,nombre_oferta,precio_mensual,ahorro_pct,gb_incluidos
19,OF020,movistar total basico,149.9,20,30
20,OF021,movistar total plus,189.9,35,60
21,OF022,movistar total max,229.9,50,9999


**Hallazgo:** las variantes MT se pueden comparar directamente porque son pocas ofertas.